Se importaron los archivos de .csv

In [2]:
import pandas as pd
import numpy as np

In [3]:
aisles = pd.read_csv("../data/raw/aisles.csv", sep=";")

departments = pd.read_csv("../data/raw/departments.csv", sep=";")

orders = pd.read_csv("../data/raw/instacart_orders.csv", sep=";")

order_prods = pd.read_csv("../data/raw/order_products.csv", sep=";")

products = pd.read_csv("../data/raw/products.csv", sep=";")

LECCION 2:LIMPIEZA Y ESTANDARIZACION

In [31]:
print(orders.columns.tolist())
print("Índice:", orders.index.name)

['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Índice: None


TIPO DE DATO CON asktype

In [30]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 26.8 MB


Después: 11.0 MB


NULOS :COMPROBAR LA CAUSA ANTES DE DESIDIR

In [29]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


In [28]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

Series([], Name: count, dtype: int64)
0


TEXTO CON LOS METODOS .STR

In [27]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 0
Espacio invisible: 0
0 0


DUPLICADOS IMPLICITOS

In [6]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


LIMPIEZA REUTILIZABLE CON FUNCIONES

In [7]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


VERIFICACION DE LA LIMPIEZA

In [19]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

LABORATORIO GUIADO 2:PIPELINE DE LIMPIEZA CON INFORME DE CAMBIO

PASO 1 :RECARGAR LOS DATOS CRUDOS

In [21]:
orders_crudo = pd.read_csv("../data/raw/instacart_orders.csv", sep=";")

products_crudo = pd.read_csv("../data/raw/products.csv", sep=";")

op_crudo = pd.read_csv("../data/raw/order_products.csv", sep=";")

print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


PASO 2:UNA FUNCION QUE ANOTA EL ESTADO DE LA TABLA

In [22]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

PASO 3:UN PASO DE LIMPIEZA POR FUNCION

In [23]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

PASO 4:ENCADENAR LOS PASOS CON PIPE

In [24]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

PASO 5:INFORME DE CAMBIOS

In [25]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,5.1
4,products,1 nombres,49694,0,5.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


PASO 6:VALIDAR LOS RESULTADOS

In [32]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
False True


RETO 2

1. Comprueba si order_prods tiene filas duplicadas exactas, si se repite algún par order_id + product_id y si reordered solo contiene 0 y 1.

In [4]:

print("Duplicados exactos:", order_prods.duplicated().sum())


print(
    "Combinaciones repetidas:",
    order_prods.duplicated(subset=["order_id", "product_id"]).sum()
)


print("Valores encontrados:", order_prods["reordered"].unique())


print("¿Solo contiene 0 y 1?:", order_prods["reordered"].isin([0, 1]).all())

Duplicados exactos: 0
Combinaciones repetidas: 0
Valores encontrados: [0 1]
¿Solo contiene 0 y 1?: True


2. Comprueba si aisles y departments tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?

In [5]:

print("Aisles duplicados:", aisles["aisle"].duplicated().sum())
print("Departments duplicados:", departments["department"].duplicated().sum())


print(
    "Aisles con espacios:",
    (aisles["aisle"] != aisles["aisle"].str.strip()).sum()
)

print(
    "Departments con espacios:",
    (departments["department"] != departments["department"].str.strip()).sum()
)


print(aisles["aisle"].unique())
print(departments["department"].unique())

Aisles duplicados: 0
Departments duplicados: 0
Aisles con espacios: 0
Departments con espacios: 0
<StringArray>
[     'prepared soups salads',          'specialty cheeses',
        'energy granola bars',              'instant foods',
 'marinades meat preparation',                      'other',
              'packaged meat',            'bakery desserts',
                'pasta sauce',           'kitchen supplies',
 ...
        'trail mix snack mix',              'feminine care',
          'body lotions soap',       'tortillas flat bread',
    'frozen appetizers sides',   'hot cereal pancake mixes',
                  'dry pasta',                     'beauty',
 'muscles joints pain relief', 'specialty wines champagnes']
Length: 134, dtype: str
<StringArray>
[         'frozen',           'other',          'bakery',         'produce',
         'alcohol',   'international',       'beverages',            'pets',
 'dry goods pasta',            'bulk',   'personal care',    'meat seafood',
    

3. Investiga los 836 nulos de add_to_cart_order: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [6]:

nulos = order_prods[order_prods["add_to_cart_order"].isna()]

print("Total de valores nulos:", len(nulos))


pedidos_afectados = nulos["order_id"].unique()

print("Cantidad de pedidos afectados:", len(pedidos_afectados))


productos_por_pedido = (
    order_prods.groupby("order_id")
    .size()
)

cantidades = productos_por_pedido.loc[pedidos_afectados]

print("Cantidad mínima de productos:", cantidades.min())
print("Cantidad máxima de productos:", cantidades.max())
print("Resumen de productos por pedido:")
print(cantidades.describe())

Total de valores nulos: 836
Cantidad de pedidos afectados: 70
Cantidad mínima de productos: 65
Cantidad máxima de productos: 127
Resumen de productos por pedido:
count     70.000000
mean      75.942857
std       12.898585
min       65.000000
25%       67.000000
50%       71.000000
75%       78.000000
max      127.000000
dtype: float64


4. Carga de nuevo products.csv en una variable products_crudo y cuenta cuántos nombres (no nulos) cambiaron con la estandarización.

In [7]:

products_crudo = pd.read_csv("../data/raw/products.csv", sep=";")


comparacion = products_crudo[
    ["product_id", "product_name"]
].merge(
    products[["product_id", "product_name"]],
    on="product_id",
    suffixes=("_original", "_limpio")
)


cambiaron = (
    comparacion["product_name_original"].notna()
    & comparacion["product_name_limpio"].notna()
    & (
        comparacion["product_name_original"]
        != comparacion["product_name_limpio"]
    )
)

print("Nombres que cambiaron:", cambiaron.sum())

Nombres que cambiaron: 0


5. Crea una Series con el nombre del día de cada pedido y conviértela a una categoría ordenada de domingo a sábado. Compara la memoria del texto con la de la categoría.

In [8]:

dias = {
    0: "Domingo",
    1: "Lunes",
    2: "Martes",
    3: "Miércoles",
    4: "Jueves",
    5: "Viernes",
    6: "Sábado"
}

dias_semana = orders["order_dow"].map(dias)


orden_dias = [
    "Domingo", "Lunes", "Martes",
    "Miércoles", "Jueves", "Viernes", "Sábado"
]

dias_categoria = pd.Categorical(
    dias_semana,
    categories=orden_dias,
    ordered=True
)


memoria_texto = dias_semana.memory_usage(deep=True)
memoria_categoria = dias_categoria.memory_usage(deep=True)

print("Memoria como texto:", memoria_texto, "bytes")
print("Memoria como categoría:", memoria_categoria, "bytes")

print("¿La categoría ahorra memoria?:", memoria_categoria < memoria_texto)

Memoria como texto: 28568422 bytes
Memoria como categoría: 479688 bytes
¿La categoría ahorra memoria?: True


6. ¿Cuántas filas de order_prods corresponden a productos 'Unknown' y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica.

In [9]:

unknown = products[products["product_name"] == "Unknown"]


ids_unknown = unknown["product_id"]


registros_unknown = order_prods[
    order_prods["product_id"].isin(ids_unknown)
]


print("Registros de productos Unknown:", len(registros_unknown))

print(
    "Pedidos afectados:",
    registros_unknown["order_id"].nunique()
)

Registros de productos Unknown: 0
Pedidos afectados: 0


7. En una copia de departments, reemplaza 'missing' por un nulo real (pd.NA). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.

In [10]:

departments_copia = departments.copy()


departments_copia["department"] = (
    departments_copia["department"].replace("missing", pd.NA)
)


print("Valores originales:")
print(departments["department"].unique())

print("\nValores después del reemplazo:")
print(departments_copia["department"].unique())

print(
    "\nCantidad de valores nulos:",
    departments_copia["department"].isna().sum()
)

Valores originales:
<StringArray>
[         'frozen',           'other',          'bakery',         'produce',
         'alcohol',   'international',       'beverages',            'pets',
 'dry goods pasta',            'bulk',   'personal care',    'meat seafood',
          'pantry',       'breakfast',    'canned goods',      'dairy eggs',
       'household',          'babies',          'snacks',            'deli',
         'missing']
Length: 21, dtype: str

Valores después del reemplazo:
<StringArray>
[         'frozen',           'other',          'bakery',         'produce',
         'alcohol',   'international',       'beverages',            'pets',
 'dry goods pasta',            'bulk',   'personal care',    'meat seafood',
          'pantry',       'breakfast',    'canned goods',      'dairy eggs',
       'household',          'babies',          'snacks',            'deli',
               nan]
Length: 21, dtype: str

Cantidad de valores nulos: 1


LECCION 3:FILTRADO , ORDENAMIENTO Y AGRUPACION DE DATOS

FILTRADO CON MASCARAS BOOLEANAS

In [5]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28819


Combinacion de condiciones

In [6]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


FILTRADO CON QUERY

In [7]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


Ordenamiento con sort_values y nlargest

In [8]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237015        2             1
317143        2             4
337248        4             2


AGRUPACION CON GROUPBY

In [9]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


AGREGACION CON NOMBRE

In [10]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60912,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


Pivot_table

In [11]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


Transform

In [12]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231960 30528


LABORATORIO GUIA 3:PERFIL DE HABITOS DE COMPRA

PASO 1:PEDIDOS POR CLIENTE

In [13]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042277
std           2.746990
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

PASO 2:SEGMENTAR CLIENTES CON PD.CUT

In [14]:
import numpy as np

segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55355
regular      71554
frecuente    30528
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


PASO 3:LLEVAR EL SEGMENTO A CADA PEDIDO CON MAP

In [15]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


PASO 4:RESUMEN POR SEGMENTO 

In [16]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55355,55355,16.0
regular,71554,191652,11.0
frecuente,30528,231960,6.0


PASO 5:FRANJAS HORARIOS Y TABLA CRUZADA

In [17]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


Paso 6: las combinaciones de día y franja más activas

In [18]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

Paso 7: dejar orders como estaba

In [19]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478967, 6)


RETO 3

1. ¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?

In [20]:
domingos = orders[orders['order_dow'] == 0]

pedidos_hora = domingos['order_hour_of_day'].value_counts()

print(pedidos_hora.head(3))

order_hour_of_day
15    7585
14    7533
13    7483
Name: count, dtype: int64


2. ¿A qué hora es mayor la espera media entre pedidos (days_since_prior_order) y a qué hora es menor?

In [21]:
espera_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()

print("Mayor espera:")
print(espera_hora.idxmax(), espera_hora.max())

print("Menor espera:")
print(espera_hora.idxmin(), espera_hora.min())

Mayor espera:
0 12.520080998987513
Menor espera:
22 10.617948077883176


3. ¿Cuántos clientes tienen 10 pedidos o más?

In [22]:
clientes = orders.groupby('user_id')['order_number'].max()

print((clientes >= 10).sum())

77093


4. ¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?

In [23]:
pedidos_7_dias = (orders['days_since_prior_order'] == 7).sum()

porcentaje = (pedidos_7_dias / len(orders)) * 100

print("Pedidos:", pedidos_7_dias)
print("Porcentaje:", porcentaje)

Pedidos: 44579
Porcentaje: 9.30732179878781


5. Usando order_prods: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?

In [24]:
productos_pedido = order_prods.groupby('order_id').size()

print("Pedidos con 20 productos o más:")
print((productos_pedido >= 20).sum())

print("Pedido más grande:")
print(productos_pedido.idxmax())

print("Cantidad de productos:")
print(productos_pedido.max())

Pedidos con 20 productos o más:
49420
Pedido más grande:
61355
Cantidad de productos:
127


6. Usando order_prods: calcula la tasa de recompra (media de reordered) para las posiciones 1 a 5 del carrito. ¿Qué patrón ves?

In [ ]:
fin_semana = orders[orders['order_dow'].isin([0, 6])]

entre_semana = orders[~orders['order_dow'].isin([0, 6])]

print("Mediana fin de semana:")
print(fin_semana['days_since_prior_order'].median())

print("Mediana entre semana:")
print(entre_semana['days_since_prior_order'].median())

7. Compara la mediana de days_since_prior_order de los pedidos de fin de semana (domingo y sábado) con la del resto de la semana.

In [25]:
fin_semana = orders[orders['order_dow'].isin([0, 6])]

entre_semana = orders[~orders['order_dow'].isin([0, 6])]

print("Mediana fin de semana:")
print(fin_semana['days_since_prior_order'].median())

print("Mediana entre semana:")
print(entre_semana['days_since_prior_order'].median())

Mediana fin de semana:
8.0
Mediana entre semana:
7.0


LECCION 4:COMBINACION Y ESTRUCTURACION DE MULTIPLES DATASET

TIPOS DE UNION CON EL PARAMETRO how

In [26]:
pedidos = pd.DataFrame({'order_id': [1, 2, 3], 'user_id': [10, 20, 30]})
detalle = pd.DataFrame({'order_id': [1, 1, 2, 4], 'product_id': [100, 200, 100, 300]})

for tipo in ['inner', 'left', 'outer']:
    resultado = pedidos.merge(detalle, on='order_id', how=tipo)
    print(tipo, len(resultado))
    print(resultado, '\n')

inner 3
   order_id  user_id  product_id
0         1       10         100
1         1       10         200
2         2       20         100 

left 4
   order_id  user_id  product_id
0         1       10       100.0
1         1       10       200.0
2         2       20       100.0
3         3       30         NaN 

outer 5
   order_id  user_id  product_id
0         1     10.0       100.0
1         1     10.0       200.0
2         2     20.0       100.0
3         3     30.0         NaN
4         4      NaN       300.0 



VALIDACION DE LA RELACION CON validate

In [27]:
catalogo_malo = pd.concat([products, products.head(1)])

try:
    order_prods.merge(catalogo_malo, on='product_id', validate='m:1')
except pd.errors.MergeError as error:
    print('Detectado:', str(error).splitlines()[0])

Detectado: Merge keys are not unique in right dataset; not a many-to-one merge


AUDITORIA DE LA UNION CON indicator

In [28]:
auditoria = orders.merge(order_prods[['order_id']].drop_duplicates(),
                         on='order_id', how='left', indicator=True)
auditoria['_merge'].value_counts()

_merge
both          450061
left_only      28906
right_only         0
Name: count, dtype: int64

CONSTRUCCION DE LA TABLA MAESTRA

In [14]:
filas_inicio = len(order_prods)

maestra = order_prods.merge(products, on='product_id', how='left', validate='m:1')
print('+ products   ', maestra.shape)
maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print('+ aisles     ', maestra.shape)
maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print('+ departments', maestra.shape)
maestra = maestra.merge(orders, on='order_id', how='left', validate='m:1')
print('+ orders     ', maestra.shape)

assert len(maestra) == filas_inicio

+ products    (4545007, 7)
+ aisles      (4545007, 8)
+ departments (4545007, 9)
+ orders      (4545007, 14)


PREGUNTAS SOBRE LA TABLA MAESTRA

In [4]:
top = (maestra.groupby(['product_id', 'product_name']).size()
              .nlargest(5).reset_index(name='veces'))

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
por_depto.head(3)

,articulos,tasa_recompra
department,,
produce,1328302,0.651140
dairy eggs,759842,0.670597
snacks,404850,0.574442


Aplicado con concat

In [5]:
categorias = pd.concat([
    aisles.rename(columns={'aisle_id': 'id', 'aisle': 'nombre'}).assign(tipo='pasillo'),
    departments.rename(columns={'department_id': 'id', 'department': 'nombre'}).assign(tipo='departamento'),
], ignore_index=True)
print(categorias.shape)

(155, 3)


Formato ancho y largo con pivot_table y melt

In [6]:
ancho = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                           values='order_id', aggfunc='count')
largo = (ancho.reset_index()
              .melt(id_vars='order_hour_of_day', var_name='order_dow', value_name='pedidos'))
print(ancho.shape, largo.shape)

(24, 7) (168, 3)


Laboratorio guiado 4: tabla de análisis por producto

Paso 1: agregar la tabla maestra por producto

In [15]:
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())
print(por_producto.shape)
por_producto.head(3)

(45573, 5)


,product_id,veces,clientes,tasa_recompra,posicion_media
0,1,280,206,0.564286,5.642857
1,2,11,11,0.000000,8.727273
2,3,42,27,0.738095,6.952381


Paso 2: recuperar nombre y departamento con merge

In [16]:
ficha_productos = (por_producto
    .merge(products, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])
print(ficha_productos.shape)
ficha_productos.loc[ficha_productos['veces'].idxmax()]

(45573, 7)


product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object

Paso 3: filtrar los productos con suficientes datos

In [17]:
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(len(populares))

739


Paso 4: clasificar dentro de cada departamento con rank

In [18]:
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))
print(len(lideres))
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)

18


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004


Paso 5: la misma respuesta con sort_values y head

In [19]:
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))
print(len(lideres_2), set(lideres_2['product_id']) == set(lideres['product_id']))


18 True


Paso 6: guardar la ficha de productos

In [26]:
ficha_productos.to_parquet(SALIDA / 'ficha_productos.parquet', index=False)
lideres.to_csv(SALIDA / 'lideres_por_departamento.csv', index=False)

RETO 4

1. Entre los pasillos con al menos 10 000 artículos pedidos, ¿cuáles son los tres con mayor tasa de recompra? ¿Qué pasillo tiene la tasa más baja de todos?

In [27]:
por_pasillo = (maestra
    .groupby('aisle')
    .agg(articulos=('order_id', 'size'),
         tasa_recompra=('reordered', 'mean'))
    .reset_index())

# Tres con mayor tasa, entre los que tienen al menos 10 000 artículos
top3_pasillos = (por_pasillo[por_pasillo['articulos'] >= 10_000]
                 .nlargest(3, 'tasa_recompra'))
print(top3_pasillos)

# Tasa más baja de todos los pasillos (sin filtro)
print(por_pasillo.nsmallest(1, 'tasa_recompra'))

                             aisle  articulos  tasa_recompra
83                            milk     125522       0.781305
131  water seltzer sparkling water     117133       0.730605
50                    fresh fruits     510101       0.718197
                 aisle  articulos  tasa_recompra
122  spices seasonings      29454       0.152407


2. Compara la participación de cada departamento en los pedidos de madrugada (0 a 5 h) con su participación total. ¿Qué departamento entra en el top 3 de madrugada pero no en el total?

In [28]:
madrugada = maestra[maestra['order_hour_of_day'].between(0, 5)]

participacion = pd.DataFrame({
    'madrugada': madrugada['department'].value_counts(normalize=True),
    'total': maestra['department'].value_counts(normalize=True),
}).sort_values('madrugada', ascending=False)
print(participacion.round(4))

top3_madrugada = set(participacion['madrugada'].nlargest(3).index)
top3_total = set(participacion['total'].nlargest(3).index)
print("Entra en el top 3 de madrugada pero no en el total:", top3_madrugada - top3_total)

                 madrugada   total
department                        
produce             0.3021  0.2923
dairy eggs          0.1587  0.1672
beverages           0.0849  0.0826
snacks              0.0789  0.0891
frozen              0.0765  0.0691
pantry              0.0617  0.0578
bakery              0.0332  0.0362
canned goods        0.0325  0.0329
deli                0.0312  0.0325
dry goods pasta     0.0272  0.0268
household           0.0256  0.0227
meat seafood        0.0209  0.0218
breakfast           0.0209  0.0219
personal care       0.0161  0.0138
babies              0.0098  0.0130
international       0.0083  0.0083
pets                0.0037  0.0030
missing             0.0029  0.0021
alcohol             0.0027  0.0048
other               0.0012  0.0011
bulk                0.0011  0.0010
Entra en el top 3 de madrugada pero no en el total: {'beverages'}


3. ¿Cuáles son los tres productos más pedidos del departamento snacks?

In [29]:
(maestra[maestra['department'] == 'snacks']
    .groupby('product_name')
    .size()
    .nlargest(3))

product_name
Lightly Salted Baked Snap Pea Crisps    3462
Original Veggie Straws                  2331
Sea Salt Pita Chips                     2275
dtype: int64

4. Construye una tabla con una fila por cliente y tres columnas: pedidos distintos, artículos y tasa de recompra. ¿Cuántos clientes tiene, cuál es el promedio de artículos por cliente y qué cliente compró más artículos?

In [30]:
por_cliente = (maestra
    .groupby('user_id')
    .agg(pedidos=('order_id', 'nunique'),
         articulos=('order_id', 'size'),
         tasa_recompra=('reordered', 'mean'))
    .reset_index())

print("Clientes:", len(por_cliente))
print("Promedio de artículos por cliente:", round(por_cliente['articulos'].mean(), 2))
print(por_cliente.nlargest(1, 'articulos'))

Clientes: 149626
Promedio de artículos por cliente: 30.38
       user_id  pedidos  articulos  tasa_recompra
75153   103634       18        580       0.822414


5. Une orders con los pedidos de order_prods usando how='inner'. ¿Cuántas filas obtienes y por qué son menos que con how='left'?

In [31]:
inner = orders.merge(order_prods, on='order_id', how='inner')
left = orders.merge(order_prods, on='order_id', how='left')
print("inner:", inner.shape, "| left:", left.shape)

# Pedidos que no tienen ningún producto
sin_productos = ~orders['order_id'].isin(order_prods['order_id'])
print("Pedidos sin productos:", sin_productos.sum())

inner: (4545007, 9) | left: (4573913, 9)
Pedidos sin productos: 28906


6. Con concat, arma una sola tabla con los tres productos más pedidos de produce y los tres de dairy eggs, con una columna que indique el departamento.

In [32]:
def top3_depto(nombre):
    return (maestra[maestra['department'] == nombre]
            .groupby('product_name')
            .size()
            .nlargest(3)
            .rename('veces')
            .reset_index()
            .assign(department=nombre))

top_produce_lacteos = pd.concat(
    [top3_depto('produce'), top3_depto('dairy eggs')],
    ignore_index=True)
top_produce_lacteos

,product_name,veces,department
0,Banana,66050,produce
1,Bag of Organic Bananas,53297,produce
2,Organic Strawberries,37039,produce
3,Organic Whole Milk,19600,dairy eggs
4,Organic Half & Half,10752,dairy eggs
5,Half & Half,9588,dairy eggs


7. Crea una tabla dinámica de artículos por departamento (filas) y día de la semana (columnas). ¿Qué forma tiene y qué día se piden más artículos de alcohol?

In [33]:
tabla = maestra.pivot_table(
    index='department',
    columns='order_dow',
    values='product_id',
    aggfunc='count',
    fill_value=0)

print(tabla.shape)
print("Día con más artículos de alcohol:", tabla.loc['alcohol'].idxmax())
tabla

(21, 7)
Día con más artículos de alcohol: 5


order_dow,0,1,2,3,4,5,6
department,,,,,,,
alcohol,2397,3025,2868,3094,3656,3814,2787
babies,11632,9994,7818,7131,7096,7053,8193
bakery,31301,28609,21583,19578,19251,21484,22935
beverages,60110,67401,51999,48375,47704,52210,47458
breakfast,17940,18104,13505,12200,12154,13137,12593
bulk,875,840,628,600,575,612,640
canned goods,31680,25104,18525,16871,16358,18080,22854
dairy eggs,142947,134533,99676,89673,89756,99780,103477
deli,29446,25896,19156,17089,16779,18881,20356
